# Building a Simple Chatbot Using Transformer Models

## Import our required tools from the transformers library

In [2]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

## Choosing a model

In [3]:
model_name = "facebook/blenderbot-400M-distill"

## Fetch the model and initialize a tokenizer

In [4]:
# Load model (download on first run and reference local installation for subsequent runs)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

pytorch_model.bin:   0%|          | 0.00/730M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/347 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/730M [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/16.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

## Keeping track of conversation history

In [6]:
conversation_history = []

In [7]:
print("Chatbot ready! (type 'exit' to quit)\n")

Chatbot ready! (type 'exit' to quit)



## Encoding the conversation history

In [8]:
history_string = "\n".join(conversation_history)

## Fetch prompt from user

In [9]:
input_text = input("> ")

## Tokenization of user prompt and chat history

In [10]:
prompt = history_string + f"\nUser: {input_text}\nBot:"

inputs = tokenizer(
    prompt,
    return_tensors="pt",
    truncation=True,
    max_length=512
)

## Generate output from the model

In [13]:
outputs = model.generate(
    **inputs,
    max_new_tokens=60,
    no_repeat_ngram_size=3,
    repetition_penalty=1.3,
    do_sample=True,
    temperature=0.6,
    top_p=0.85,
    use_cache=False
)
## Remove this print statement after testing
print(outputs)


tensor([[   1, 6950,    8,  855,  366,  304, 1752,   38,  281,  632,  404, 6231,
          278,  745,  265,  816,  704,  403,  672,   21,    2]])


## Decode output

In [14]:
response = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()
print(response)

Hello! How are you today? I am just relaxing after a long day at work.


## Update conversation history

In [15]:
conversation_history.append(f"User: {input_text}")
conversation_history.append(f"Bot: {response}")
print(conversation_history)

['User: Hello', 'Bot: Hello! How are you today? I am just relaxing after a long day at work.']


## Repeat

In [16]:
# keep only last few exchanges (prevents confusion)
conversation_history = conversation_history[-6:]

In [ ]:
# 1. Get the model's device before entering the loop
device = next(model.parameters()).device

while True:
    # keep only last few exchanges (prevents confusion)
    conversation_history = conversation_history[-6:]
    
    history_string = "\n".join(conversation_history)

    input_text = input("> ")
    if input_text.lower() == "exit":
        break

    prompt = history_string + f"\nUser: {input_text}\nBot:"

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )
    
    inputs = {k: v.to(device) for k, v in inputs.items()}

    outputs = model.generate(
        **inputs,
        max_new_tokens=60,
        no_repeat_ngram_size=3,
        repetition_penalty=1.3,
        do_sample=True,
        temperature=0.6,
        top_p=0.85,
        use_cache=False
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()
    print("Bot:", response)

    conversation_history.append(f"User: {input_text}")
    conversation_history.append(f"Bot: {response}")

Bot: Hello, how are you? What do you do for a living? I'm an accountant.
Bot: I'm doing well. Just got off work from my accounting job. What are you up to?
Bot: I am doing well, thank you for asking. What is your favorite thing to do in your spare time?
